In [ ]:
# @title **CELDA 2: ANÁLISIS DE DATOS Y TABLA RESUMEN**

import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("📊 CELDA 2: ANÁLISIS DE DATOS Y TABLA RESUMEN")
print("="*80)

# ===============================
# CONFIGURACIÓN
# ===============================
INPUT_DIR = "input"

# ===============================
# 1. CARGAR DATOS
# ===============================
print("\n📂 1. Cargando datos...")

# Cargar todos los datasets
df_inv_2024 = pd.read_csv(f"{INPUT_DIR}/Inventario_2024.csv")
df_ingresos = pd.read_csv(f"{INPUT_DIR}/Ingreso_2025.csv")
df_salidas = pd.read_csv(f"{INPUT_DIR}/Salida_2025.csv")
df_inv_teorico = pd.read_csv(f"{INPUT_DIR}/Inventario_Teorico_2025.csv")
df_inv_fisico = pd.read_csv(f"{INPUT_DIR}/Inventario_Fisico_2025.csv")
df_comparacion = pd.read_csv(f"{INPUT_DIR}/Comparacion_Inventarios.csv")
df_errores = pd.read_csv(f"{INPUT_DIR}/Log_Errores_Ingresos.csv")
lista_maestra = pd.read_csv("Lista Maestra de Residuos.csv")

print(f"   ✅ Inventario 2024: {len(df_inv_2024)} registros")
print(f"   ✅ Ingresos 2025: {len(df_ingresos)} registros")
print(f"   ✅ Salidas 2025: {len(df_salidas)} registros")
print(f"   ✅ Inventario Teórico 2025: {len(df_inv_teorico)} residuos")
print(f"   ✅ Inventario Físico 2025: {len(df_inv_fisico)} residuos")

# ===============================
# 2. ANÁLISIS ESTADÍSTICO BÁSICO
# ===============================
print("\n📈 2. Análisis estadístico básico...")

# 2.1 Totales generales
total_inv_2024 = df_inv_2024["Cantidad_Registrada_kg"].sum()
total_ingresos = df_ingresos["Cantidad_kg"].sum()
total_salidas = df_salidas["Cantidad_kg"].sum()
total_teorico = df_inv_teorico["Cantidad_kg"].sum()
total_fisico = df_inv_fisico["Cantidad_kg"].sum()
diferencia_total = total_teorico - total_fisico

print(f"   • Inventario 2024: {total_inv_2024:,.0f} kg")
print(f"   • Ingresos 2025: {total_ingresos:,.0f} kg")
print(f"   • Salidas 2025: {total_salidas:,.0f} kg")
print(f"   • Inventario Teórico 2025: {total_teorico:,.0f} kg")
print(f"   • Inventario Físico 2025: {total_fisico:,.0f} kg")
print(f"   • Diferencia Total: {diferencia_total:+,.0f} kg")

# 2.2 Análisis por tipo de residuo
print("\n📊 2.2 Análisis por tipo de residuo:")
for tipo in ["Peligroso", "No Peligroso"]:
    residuos_tipo = lista_maestra[lista_maestra["Tipo"] == tipo]["Residuo"]

    inv_tipo = df_inv_2024[df_inv_2024["Residuo"].isin(residuos_tipo)]["Cantidad_Registrada_kg"].sum()
    ing_tipo = df_ingresos[df_ingresos["Residuo"].isin(residuos_tipo)]["Cantidad_kg"].sum()
    sal_tipo = df_salidas[df_salidas["Residuo"].isin(residuos_tipo)]["Cantidad_kg"].sum()
    teo_tipo = df_inv_teorico[df_inv_teorico["Residuo"].isin(residuos_tipo)]["Cantidad_kg"].sum()
    fis_tipo = df_inv_fisico[df_inv_fisico["Residuo"].isin(residuos_tipo)]["Cantidad_kg"].sum()

    print(f"\n   {tipo}:")
    print(f"     • Stock 2024: {inv_tipo:,.0f} kg")
    print(f"     • Ingresos: {ing_tipo:,.0f} kg")
    print(f"     • Salidas: {sal_tipo:,.0f} kg")
    print(f"     • Teórico 2025: {teo_tipo:,.0f} kg")
    print(f"     • Físico 2025: {fis_tipo:,.0f} kg")
    print(f"     • Diferencia: {teo_tipo - fis_tipo:+,.0f} kg")

# 2.3 Análisis de errores
print("\n🔍 2.3 Análisis de errores identificados:")
if 'df_errores' in locals() and len(df_errores) > 0:
    print(f"   • Total errores registrados: {len(df_errores)}")

    # Errores por tipo
    errores_por_tipo = df_errores["Tipo_Error"].value_counts()
    for tipo, cantidad in errores_por_tipo.items():
        porcentaje = cantidad / len(df_errores) * 100
        print(f"   • {tipo}: {cantidad} eventos ({porcentaje:.1f}%)")

    # Errores más significativos
    error_max = df_errores["Error_kg"].abs().max()
    error_promedio = df_errores["Error_kg"].abs().mean()
    print(f"\n   • Error máximo: {error_max:,.0f} kg")
    print(f"   • Error promedio: {error_promedio:,.0f} kg")

    # Residuos con más errores
    residuos_con_errores = df_errores["Residuo_Real"].value_counts().head(5)
    print(f"\n   • Residuos con más errores:")
    for residuo, count in residuos_con_errores.items():
        print(f"     - {residuo}: {count} errores")

# ===============================
# 3. GENERAR TABLA RESUMEN DETALLADA
# ===============================
print("\n📋 3. Generando tabla resumen detallada...")

# Preparar datos para la tabla
tabla_resumen = []

for residuo in lista_maestra["Residuo"].unique():
    # Stock 2024 (usar cantidad registrada - con errores)
    stock_2024 = df_inv_2024[df_inv_2024["Residuo"] == residuo]["Cantidad_Registrada_kg"].sum()

    # Ingresos 2025
    ingresos_2025 = df_ingresos[df_ingresos["Residuo"] == residuo]["Cantidad_kg"].sum()

    # Salidas 2025
    salidas_2025 = df_salidas[df_salidas["Residuo"] == residuo]["Cantidad_kg"].sum()

    # Teórico 2025
    teorico_2025 = df_inv_teorico[df_inv_teorico["Residuo"] == residuo]["Cantidad_kg"].sum()

    # Físico 2025
    fisico_2025 = df_inv_fisico[df_inv_fisico["Residuo"] == residuo]["Cantidad_kg"].sum()

    # Diferencia
    diferencia = teorico_2025 - fisico_2025

    # Determinar tipo de diferencia
    if teorico_2025 < 0:
        tipo_diferencia = "BALANCE NEGATIVO ⚠️"
        simbolo = "⚠️"
    elif diferencia > 100:
        tipo_diferencia = "FÍSICO < TEÓRICO (SUBESTIMACIÓN) ▲"
        simbolo = "▲"
    elif diferencia < -100:
        tipo_diferencia = "FÍSICO > TEÓRICO (SOBREESTIMACIÓN) ▼"
        simbolo = "▼"
    else:
        tipo_diferencia = "SIN DIFERENCIA SIGNIFICATIVA"
        simbolo = "✓"

    # Contar errores para este residuo
    errores_residuo = 0
    if 'df_errores' in locals():
        errores_residuo = len(df_errores[df_errores["Residuo_Real"] == residuo])

    # Obtener tipo de residuo
    tipo_residuo = lista_maestra[lista_maestra["Residuo"] == residuo]["Tipo"].iloc[0]

    tabla_resumen.append({
        "Residuo": residuo,
        "Tipo": tipo_residuo,
        "Stock 2024 (kg)": round(stock_2024, 2),
        "Ingreso 2025 (kg)": round(ingresos_2025, 2),
        "Salida 2025 (kg)": round(salidas_2025, 2),
        "Teórico 2025 (kg)": round(teorico_2025, 2),
        "Físico 2025 (kg)": round(fisico_2025, 2),
        "Diferencia (kg)": round(diferencia, 2),
        "Símbolo": simbolo,
        "Tipo Diferencia": tipo_diferencia,
        "Errores Registrados": errores_residuo
    })

# Crear DataFrame de la tabla resumen
df_tabla_resumen = pd.DataFrame(tabla_resumen)

# Guardar tabla resumen completa
df_tabla_resumen.to_csv(f"{INPUT_DIR}/Tabla_Resumen_Completa.csv", index=False)
print(f"   ✅ Tabla resumen guardada: {len(df_tabla_resumen)} residuos")

# ===============================
# 4. MOSTRAR TABLA RESUMEN (VISTA RESUMIDA)
# ===============================
print("\n" + "="*150)
print("📊 TABLA RESUMEN - BALANCE DE RESIDUOS 2024-2025")
print("="*150)

# Crear una versión formateada para visualización
df_vista = df_tabla_resumen.copy()

# Formatear columnas numéricas
for col in ["Stock 2024 (kg)", "Ingreso 2025 (kg)", "Salida 2025 (kg)",
            "Teórico 2025 (kg)", "Físico 2025 (kg)", "Diferencia (kg)"]:
    df_vista[col] = df_vista[col].apply(lambda x: f"{x:,.0f}kg")

# Crear columna combinada para diferencia
df_vista["Diferencia Visual"] = df_vista.apply(
    lambda x: f"{x['Diferencia (kg)']} {x['Símbolo']}", axis=1
)

# Seleccionar columnas para mostrar
columnas_mostrar = ["Residuo", "Tipo", "Stock 2024 (kg)", "Ingreso 2025 (kg)",
                   "Salida 2025 (kg)", "Teórico 2025 (kg)", "Físico 2025 (kg)",
                   "Diferencia Visual", "Tipo Diferencia", "Errores Registrados"]

# Mostrar primeros 20 registros
print("\nVista de muestra (primeros 20 residuos):")
print("-" * 150)

# Crear encabezado formateado
encabezado = f"{'Residuo':<30} {'Tipo':<12} {'Stock 2024':>12} {'Ingreso 2025':>12} "
encabezado += f"{'Salida 2025':>12} {'Teórico 2025':>13} {'Físico 2025':>12} "
encabezado += f"{'Diferencia':>12} {'Estado':<30} {'Errores':>8}"
print(encabezado)
print("-" * 150)

# Mostrar cada fila formateada
for i, row in df_tabla_resumen.head(20).iterrows():
    # Determinar símbolo y color basado en la diferencia
    diff = row["Diferencia (kg)"]
    if row["Teórico 2025 (kg)"] < 0:
        simbolo = "⚠️"
        estado = "BALANCE NEGATIVO"
    elif diff > 100:
        simbolo = "▲"
        estado = "Físico < Teórico"
    elif diff < -100:
        simbolo = "▼"
        estado = "Físico > Teórico"
    else:
        simbolo = "✓"
        estado = "OK"

    # Formatear la línea
    linea = f"{row['Residuo'][:28]:<30} {row['Tipo'][:10]:<12} "
    linea += f"{row['Stock 2024 (kg)']:>11.0f}kg {row['Ingreso 2025 (kg)']:>11.0f}kg "
    linea += f"{row['Salida 2025 (kg)']:>11.0f}kg {row['Teórico 2025 (kg)']:>12.0f}kg "
    linea += f"{row['Físico 2025 (kg)']:>11.0f}kg {diff:>+11.0f}kg{simbolo:<2} "
    linea += f"{estado:<30} {row['Errores Registrados']:>7}"
    print(linea)

# Mostrar resumen de estadísticas
print("\n" + "="*150)
print("📈 RESUMEN ESTADÍSTICO")
print("-" * 150)

# Calcular estadísticas
total_residuos = len(df_tabla_resumen)
balance_negativo = len(df_tabla_resumen[df_tabla_resumen["Teórico 2025 (kg)"] < 0])
diferencia_significativa = len(df_tabla_resumen[df_tabla_resumen["Diferencia (kg)"].abs() > 100])
sin_diferencia = total_residuos - diferencia_significativa - balance_negativo

total_errores = df_tabla_resumen["Errores Registrados"].sum()
residuos_con_errores = len(df_tabla_resumen[df_tabla_resumen["Errores Registrados"] > 0])

print(f"• Total de residuos analizados: {total_residuos}")
print(f"• Residuos con balance negativo: {balance_negativo} ({balance_negativo/total_residuos*100:.1f}%)")
print(f"• Residuos con diferencia > 100kg: {diferencia_significativa} ({diferencia_significativa/total_residuos*100:.1f}%)")
print(f"• Residuos sin diferencia significativa: {sin_diferencia} ({sin_diferencia/total_residuos*100:.1f}%)")
print(f"• Total eventos con errores registrados: {total_errores}")
print(f"• Residuos con al menos un error: {residuos_con_errores} ({residuos_con_errores/total_residuos*100:.1f}%)")

# Top 5 residuos con mayores discrepancias
print(f"\n🔝 TOP 5 RESIDUOS CON MAYORES DISCREPANCIAS:")
top_5_discrepancias = df_tabla_resumen.nlargest(5, "Diferencia (kg)", keep='all')
for idx, row in top_5_discrepancias.iterrows():
    print(f"   {row['Residuo'][:25]:<25} {row['Diferencia (kg)']:>+10,.0f} kg ({row['Tipo Diferencia']})")

# Top 5 balances negativos
balances_negativos = df_tabla_resumen[df_tabla_resumen["Teórico 2025 (kg)"] < 0]
if len(balances_negativos) > 0:
    print(f"\n⚠️  BALANCES NEGATIVOS ({len(balances_negativos)} residuos):")
    for idx, row in balances_negativos.nsmallest(5, "Teórico 2025 (kg)").iterrows():
        print(f"   {row['Residuo'][:25]:<25} {row['Teórico 2025 (kg)']:>10,.0f} kg (Físico: {row['Físico 2025 (kg)']:,.0f} kg)")

# ===============================
# 5. GUARDAR REPORTES
# ===============================
print("\n💾 5. Guardando reportes...")

# 5.1 Reporte de balances negativos
df_balances_negativos = df_tabla_resumen[df_tabla_resumen["Teórico 2025 (kg)"] < 0].copy()
if len(df_balances_negativos) > 0:
    df_balances_negativos.to_csv(f"{INPUT_DIR}/Reporte_Balances_Negativos.csv", index=False)
    print(f"   ✅ Reporte balances negativos: {len(df_balances_negativos)} residuos")

# 5.2 Reporte de diferencias significativas (>1000kg)
df_diferencias_significativas = df_tabla_resumen[df_tabla_resumen["Diferencia (kg)"].abs() > 1000].copy()
if len(df_diferencias_significativas) > 0:
    df_diferencias_significativas.to_csv(f"{INPUT_DIR}/Reporte_Diferencias_Significativas.csv", index=False)
    print(f"   ✅ Reporte diferencias >1000kg: {len(df_diferencias_significativas)} residuos")

# 5.3 Reporte por tipo de residuo
reporte_tipo = df_tabla_resumen.groupby("Tipo").agg({
    "Residuo": "count",
    "Stock 2024 (kg)": "sum",
    "Ingreso 2025 (kg)": "sum",
    "Salida 2025 (kg)": "sum",
    "Teórico 2025 (kg)": "sum",
    "Físico 2025 (kg)": "sum",
    "Errores Registrados": "sum"
}).reset_index()

reporte_tipo["Diferencia Total (kg)"] = reporte_tipo["Teórico 2025 (kg)"] - reporte_tipo["Físico 2025 (kg)"]
reporte_tipo.to_csv(f"{INPUT_DIR}/Reporte_Por_Tipo_Residuo.csv", index=False)
print(f"   ✅ Reporte por tipo de residuo: {len(reporte_tipo)} categorías")

# 5.4 Resumen ejecutivo
resumen_ejecutivo = {
    "Fecha_Reporte": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "Total_Residuos": total_residuos,
    "Total_Stock_2024_kg": df_tabla_resumen["Stock 2024 (kg)"].sum(),
    "Total_Ingresos_2025_kg": df_tabla_resumen["Ingreso 2025 (kg)"].sum(),
    "Total_Salidas_2025_kg": df_tabla_resumen["Salida 2025 (kg)"].sum(),
    "Total_Teorico_2025_kg": df_tabla_resumen["Teórico 2025 (kg)"].sum(),
    "Total_Fisico_2025_kg": df_tabla_resumen["Físico 2025 (kg)"].sum(),
    "Diferencia_Total_kg": df_tabla_resumen["Diferencia (kg)"].sum(),
    "Residuos_Balance_Negativo": balance_negativo,
    "Residuos_Diferencia_Significativa": diferencia_significativa,
    "Total_Eventos_Error": total_errores,
    "Residuos_Con_Errores": residuos_con_errores
}

df_resumen_ejecutivo = pd.DataFrame([resumen_ejecutivo])
df_resumen_ejecutivo.to_csv(f"{INPUT_DIR}/Resumen_Ejecutivo.csv", index=False)
print(f"   ✅ Resumen ejecutivo guardado")

print("\n" + "="*150)
print("📋 RESUMEN DE ARCHIVOS GENERADOS:")
print("-" * 150)
print("1. Tabla_Resumen_Completa.csv - Tabla completa de todos los residuos")
print("2. Reporte_Balances_Negativos.csv - Residuos con balance negativo")
print("3. Reporte_Diferencias_Significativas.csv - Diferencias > 1000kg")
print("4. Reporte_Por_Tipo_Residuo.csv - Agregado por tipo de residuo")
print("5. Resumen_Ejecutivo.csv - Métricas clave del análisis")

print("\n" + "="*150)
print("✅ CELDA 2 COMPLETADA EXITOSAMENTE")
print("   Se han analizado todos los datos y generado tablas resumen.")
print("   Proceda a ejecutar la CELDA 3 para aplicar método FIFO Físico.")
print("="*150)

# Mostrar vista previa de la tabla completa (primeras 5 filas)
print("\n📋 VISTA PREVIA TABLA COMPLETA (primeras 5 filas):")
print(df_tabla_resumen[["Residuo", "Tipo", "Stock 2024 (kg)", "Ingreso 2025 (kg)",
                       "Salida 2025 (kg)", "Teórico 2025 (kg)", "Físico 2025 (kg)",
                       "Diferencia (kg)", "Tipo Diferencia"]].head().to_string(index=False))